<a href="https://colab.research.google.com/github/JittoJoseph/GaussianSplat-Colab/blob/main/Video_to_GaussianSplat_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Walkthrough Video to 3D Gaussian Splat

Smartphone walkthrough video, in. Downloadable 3D Gaussian Splat (`.ply` + `.splat`), out. All in Colab.

**Pipeline:** frames, then Structure-from-Motion for camera poses (COLMAP / hloc), then
[Splatfacto](https://docs.nerf.studio/nerfology/methods/splat.html) optimization (Nerfstudio + `gsplat`),
then export. No pretrained weights; the splat is fit to *your* scene.

*Chosen over gsplat-standalone, DN-Splatter, OpenSplat, and raw 3DGS/2DGS for the best
reliability-per-quality: actively maintained, fully automated, and it embeds gsplat's MCMC
engine + camera-pose refinement. DN-Splatter is noted at the end as an indoor-geometry upgrade.*

> **First:** `Runtime -> Change runtime type -> GPU` (T4 works; L4/A100 are faster). Then run cells top to bottom.


## 1. GPU check
Confirms a CUDA GPU before doing anything expensive.

In [13]:
#@title Verify GPU { display-mode: "form" }
import subprocess, sys, textwrap

try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"], text=True).strip()
    name, mem, driver = [x.strip() for x in out.split(",")]
    GPU_MEM_GB = float(mem.split()[0]) / 1024.0
    print(f"GPU     : {name}")
    print(f"VRAM    : {mem}   Driver: {driver}")
    print(f"Python  : {sys.version.split()[0]}")
    if GPU_MEM_GB < 14:
        print("\nNote: <16 GB VRAM (typical free T4). Prefer the 'balanced' preset\n"
              "and keep target_frames moderate to avoid out-of-memory.")
    else:
        print("\nPlenty of VRAM, 'high' or 'max' preset is fine.")
except Exception:
    GPU_MEM_GB = 0.0
    print(textwrap.dedent("""
        NO GPU DETECTED.
        Runtime -> Change runtime type -> Hardware accelerator -> GPU, then re-run.
        (Changing it restarts the runtime, so run this cell again after.)"""))
    raise SystemExit("Stop until a GPU is available.")

GPU     : Tesla T4
VRAM    : 15360 MiB   Driver: 580.82.07
Python  : 3.12.13

Plenty of VRAM, 'high' or 'max' preset is fine.


## 2. Settings

- **quality_preset** , `balanced` (fast, ~6 GB) / `high` = `splatfacto-big` (recommended, ~12 GB) /
  `max` = official `splatfacto-mcmc` (best detail).
- **train_iterations** , higher = better + slower.
- **target_frames** , 100 to 300 is a good range for a walkthrough.
- **sfm_tool** , `colmap` (reliable default) or `hloc` (SuperPoint+SuperGlue, better for
  shaky / low-texture phone video).


In [ ]:
#@title Pipeline settings { display-mode: "form" }
quality_preset   = "high"     #@param ["balanced", "high", "max"]
train_iterations = 30000      #@param {type:"slider", min:5000, max:60000, step:1000}
target_frames    = 200        #@param {type:"slider", min:50, max:500, step:10}
sfm_tool         = "colmap"   #@param ["colmap", "hloc"]
scene_name       = "my_scene" #@param {type:"string"}

import re, os
scene_name = re.sub(r"[^A-Za-z0-9_\-]", "_", scene_name) or "my_scene"
WORK       = "/content/gsplat_project"
VIDEO_DIR  = f"{WORK}/video"
DATA_DIR   = f"{WORK}/processed/{scene_name}"
OUTPUT_DIR = f"{WORK}/outputs"
EXPORT_DIR = f"{WORK}/exports/{scene_name}"
for d in (VIDEO_DIR, DATA_DIR, OUTPUT_DIR, EXPORT_DIR):
    os.makedirs(d, exist_ok=True)
print(f"{quality_preset=}  {train_iterations=}  {target_frames=}  {sfm_tool=}  {scene_name=}")

## 3. System tools
Installs COLMAP + FFmpeg (~1 to 2 min).

In [ ]:
#@title Install COLMAP + FFmpeg { display-mode: "form" }
import subprocess, sys, os

def run(cmd, desc=None, log_path=None, ok_msg=None):
    """Run a shell command with a compact live status line; raise on failure."""
    if desc: print(f"-> {desc}")
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    tail, logf = [], open(log_path, "w") if log_path else None
    for line in proc.stdout:
        if logf: logf.write(line)
        tail.append(line.rstrip()); tail = tail[-400:]
        sys.stdout.write("\r   " + line.rstrip()[:100].ljust(102)); sys.stdout.flush()
    proc.wait()
    if logf: logf.close()
    sys.stdout.write("\r" + " " * 106 + "\r"); sys.stdout.flush()
    if proc.returncode != 0:
        print(f"   FAILED: {desc or cmd}\n   --- last lines " + "-" * 30)
        for l in tail[-25:]: print("   " + l)
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {cmd}")
    print(f"   {ok_msg or 'done'}")
    return "\n".join(tail)

run("apt-get -qq update", "apt update")
run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y colmap ffmpeg xvfb",
    "colmap + ffmpeg + xvfb")
for tool in ("colmap", "ffmpeg"):
    v = subprocess.run([tool, "-h" if tool == "colmap" else "-version"],
                       capture_output=True, text=True)
    print(f"   {tool}: {(v.stdout or v.stderr).splitlines()[0][:55]}")
print("System tools ready.")

## 4. Install Nerfstudio + gsplat
The main install (~3 to 6 min). `gsplat` compiles CUDA kernels on first use, so we warm it up
here to catch build issues early. tiny-cuda-nn is skipped (Splatfacto doesn't need it).

In [ ]:
#@title Install Nerfstudio + gsplat { display-mode: "form" }
import os, subprocess, sys
try:
    import torch
    cap = torch.cuda.get_device_capability()
    os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cap[0]}.{cap[1]}"
    print(f"torch {torch.__version__}  CUDA {torch.version.cuda}  arch {cap[0]}.{cap[1]}")
except Exception as e:
    print("torch not importable yet:", e)

run("pip -q install nerfstudio", "nerfstudio + gsplat (please wait)",
    log_path="/content/install_nerfstudio.log")

print("Warming up gsplat CUDA kernels (first compile: 1 to 3 min)...")
warm = subprocess.run(
    [sys.executable, "-c",
     "import warnings; warnings.filterwarnings('ignore');"
     "import gsplat; from gsplat import rasterization; print('gsplat', gsplat.__version__)"],
    capture_output=True, text=True)
print("  " + (warm.stdout.strip() or warm.stderr[-800:]))
if warm.returncode != 0:
    print("  If this is a compile error, re-run this cell once (kernels cache).")

cli = subprocess.run(["ns-train", "--help"], capture_output=True, text=True)
if cli.returncode != 0:
    raise RuntimeError("ns-train not on PATH, check /content/install_nerfstudio.log")
print("Nerfstudio CLI ready (ns-process-data / ns-train / ns-export).")

## 5. Provide your video

Pick **one** `input_method`:
- **`upload`** , a "Choose Files" button appears below; select the video.
- **`google_drive_link`** , paste a Drive **share link** (shared "Anyone with the link");
  downloaded via `gdown`.
- **`direct_url_or_path`** , a direct `https://...` video URL, or a local/Drive-mounted file path.

Types: `.mp4 .mov .m4v .avi .mkv .webm`. Capture tip: walk slowly, good light, overlap views, 30 to 90 s.


In [ ]:
#@title Get the video: upload / Drive link / URL or path { display-mode: "form" }
import os, sys, shutil, subprocess, json
input_method = "upload"  #@param ["upload", "google_drive_link", "direct_url_or_path"]
#@markdown Paste here for google_drive_link / direct_url_or_path (leave blank for upload):
link_or_url = ""  #@param {type:"string"}

# To use a Google Drive *path*: uncomment, run, then pick direct_url_or_path + set the path.
# from google.colab import drive; drive.mount('/content/drive')

VIDEO_EXTS = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")
VIDEO_PATH = None

if input_method == "upload":
    from google.colab import files
    print("Click 'Choose Files' and pick your walkthrough video...\n")
    up = files.upload()
    if not up: raise RuntimeError("No file selected. Re-run and choose a video.")
    fn = list(up.keys())[0]
    if not fn.lower().endswith(VIDEO_EXTS):
        raise ValueError(f"'{fn}' is not a video type {VIDEO_EXTS}.")
    VIDEO_PATH = os.path.join(VIDEO_DIR, fn)
    open(VIDEO_PATH, "wb").write(up[fn])
else:
    src = link_or_url.strip()
    if not src:
        raise ValueError("link_or_url is empty (or set input_method='upload').")
    if input_method == "google_drive_link" or "drive.google.com" in src or "docs.google.com" in src:
        try: import gdown
        except Exception:
            subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gdown"]); import gdown
        print("Downloading from Google Drive...\n")
        got = gdown.download(url=src, output=os.path.join(VIDEO_DIR, "walkthrough_input"),
                             quiet=False, fuzzy=True)
        if not got or not os.path.isfile(got):
            raise RuntimeError("gdown failed, ensure the link is shared 'Anyone with the link' "
                               "and points to a single file (not a folder).")
        VIDEO_PATH = got
    elif src.startswith(("http://", "https://")):
        ext = os.path.splitext(src.split("?")[0])[1].lower() or ".mp4"
        VIDEO_PATH = os.path.join(VIDEO_DIR, "walkthrough_input" + ext)
        run(f"wget -q --show-progress -O '{VIDEO_PATH}' '{src}'", "Downloading video")
        if os.path.getsize(VIDEO_PATH) < 1024:
            raise RuntimeError("Downloaded file is tiny/empty, URL may not be a direct video link.")
    elif os.path.isfile(src):
        VIDEO_PATH = os.path.join(VIDEO_DIR, os.path.basename(src)); shutil.copy(src, VIDEO_PATH)
    else:
        raise FileNotFoundError(f"'{src}' is neither a URL nor an existing file path.")

# Quick probe for immediate feedback.
r = subprocess.run("ffprobe -v error -select_streams v:0 -show_entries "
                   "stream=width,height:format=duration -of json " + f"'{VIDEO_PATH}'",
                   shell=True, capture_output=True, text=True)
print(f"\nVideo ready: {os.path.basename(VIDEO_PATH)}  "
      f"({os.path.getsize(VIDEO_PATH)/1e6:.1f} MB)")
try:
    j = json.loads(r.stdout); st = j["streams"][0]
    dur = float(j["format"]["duration"])
    print(f"  {st['width']}x{st['height']}, {dur:.1f} s")
    if dur < 5: print("  WARNING: very short (<5 s), reconstruction may be poor.")
except Exception:
    print("  (details unavailable, but the file will still be processed)")

## 6. Recover camera poses (Structure-from-Motion)

The modern replacement for manual calibration: `ns-process-data` extracts frames and runs SfM to
recover per-frame camera poses + a sparse point cloud (used to initialize the Gaussians).
Usually the **slowest** stage. We verify a valid `transforms.json` at the end.


In [ ]:
#@title Run SfM { display-mode: "form" }
import os, json, glob, shutil
if os.path.isdir(DATA_DIR) and os.listdir(DATA_DIR):
    shutil.rmtree(DATA_DIR); os.makedirs(DATA_DIR, exist_ok=True)

# COLMAP GPU SIFT needs an X server on headless Colab -> xvfb; hloc doesn't.
prefix = "" if sfm_tool == "hloc" else "xvfb-run -a "
match  = "sequential" if sfm_tool == "colmap" else "exhaustive"
cmd = (f"{prefix}ns-process-data video --data '{VIDEO_PATH}' --output-dir '{DATA_DIR}' "
       f"--num-frames-target {target_frames} --sfm-tool {sfm_tool} "
       f"--matching-method {match} --verbose")
print(cmd, "\n")
try:
    run(cmd, "Structure-from-Motion", log_path="/content/ns_process.log", ok_msg="SfM finished")
except RuntimeError:
    print("\nSfM failed. Try: sfm_tool='hloc' (cell 2), recapture with slower motion / more\n"
          "overlap, or raise target_frames.")
    raise

tj = os.path.join(DATA_DIR, "transforms.json")
if not os.path.isfile(tj):
    raise RuntimeError("No transforms.json, SfM produced no valid poses.")
T = json.load(open(tj))
n_frames = len(T.get("frames", []))
n_imgs = len(glob.glob(os.path.join(DATA_DIR, "images", "*")))
has_pts = os.path.isfile(os.path.join(DATA_DIR, "sparse_pc.ply")) or bool(
    glob.glob(os.path.join(DATA_DIR, "colmap", "sparse", "**", "points3D*"), recursive=True))
print(f"\nregistered frames: {n_frames} / {n_imgs} images   sparse points: {'yes' if has_pts else 'no'}")
if n_frames < 20:
    print("WARNING: <20 frames registered, quality will be low. Try sfm_tool='hloc' or recapture.")
else:
    print("Poses look good. Ready to train.")

## 7. Train the Gaussian Splat

Splatfacto fits the Gaussians to your images (no pretrained weights). Presets:

| preset | method | notes |
|---|---|---|
| `balanced` | `splatfacto` | fastest |
| `high` | `splatfacto-big` | more Gaussians, higher quality (recommended) |
| `max` | `splatfacto-mcmc` | gsplat MCMC densification, best detail |

All presets add **camera-pose refinement** (`SO3xR3`) and **bilateral-grid exposure correction**,
which help handheld phone video with auto-exposure. Progress lines stream below; the full log is saved.


In [ ]:
#@title Train { display-mode: "form" }
import os, glob, subprocess

# Flags shared by all presets. Bilateral grid + pose refinement help phone video.
common = [
    f"--data '{DATA_DIR}'", f"--output-dir '{OUTPUT_DIR}'",
    f"--max-num-iterations {train_iterations}",
    "--viewer.quit-on-train-completion True", "--vis tensorboard",
    f"--experiment-name {scene_name}",
    "--pipeline.model.camera-optimizer.mode SO3xR3",
    "--pipeline.model.use-bilateral-grid True",
]
# method name + preset-specific extras (splatfacto-big already sets cull_alpha_thresh=0.005).
PRESETS = {
    "balanced": ("splatfacto",      []),
    "high":     ("splatfacto-big",  []),
    "max":      ("splatfacto-mcmc", ["--pipeline.model.max-gs-num 3000000"]),
}
method, extra = PRESETS[quality_preset]

def build(method, extra):
    return (f"ns-train {method} " + " ".join(common + extra) +
            f" nerfstudio-data --data '{DATA_DIR}'")

def train(method, extra):
    cmd = build(method, extra); print("->", cmd, "\n")
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    log = "/content/ns_train.log"
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    with open(log, "w") as lf:
        for line in p.stdout:
            lf.write(line); s = line.rstrip()
            if any(k in s for k in ("Step (", "ETA", "out of memory", "Traceback",
                                    "Error", "Saving", "Unrecognized")):
                print(s[:120])
    p.wait(); return p.returncode, log

print(f"Training [{quality_preset}] -> '{method}', {train_iterations} iters\n")
rc, log = train(method, extra)
if rc != 0:
    tail = open(log).read()[-1500:]
    if "out of memory" in tail.lower():
        raise RuntimeError("CUDA OOM. Use 'balanced', lower target_frames, or a bigger GPU (L4/A100).")
    if quality_preset == "max":
        print("\n'max' unavailable on this build; retrying 'high' (splatfacto-big)...\n")
        rc, log = train("splatfacto-big", [])
    if rc != 0:
        print("\n--- log tail ---\n", tail)
        raise RuntimeError("Training failed. See /content/ns_train.log")

cfgs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "config.yml"), recursive=True),
              key=os.path.getmtime)
if not cfgs: raise RuntimeError("Training finished but no config.yml found.")
CONFIG_YML = cfgs[-1]
print("\nTraining complete.\nconfig:", CONFIG_YML)

## 8. Export
Lossless **`.ply`** (best quality; SuperSplat / PlayCanvas / Three.js) + a compact **`.splat`** for web viewers.

In [ ]:
#@title Export .ply (+ .splat) { display-mode: "form" }
import os, glob, numpy as np
run(f"ns-export gaussian-splat --load-config '{CONFIG_YML}' --output-dir '{EXPORT_DIR}'",
    "Exporting .ply", log_path="/content/ns_export.log")
plys = glob.glob(os.path.join(EXPORT_DIR, "*.ply"))
if not plys: raise RuntimeError("No .ply produced, see /content/ns_export.log")
PLY_PATH = plys[0]
print(f"  .ply: {PLY_PATH}  ({os.path.getsize(PLY_PATH)/1e6:.1f} MB)")

def ply_to_splat(ply_path, splat_path):
    # antimatter15 .splat: pos(3 f32) scale(3 f32) rgba(4 u8) rot-quat(4 u8) = 32 bytes.
    from plyfile import PlyData
    v = PlyData.read(ply_path)["vertex"]
    xyz = np.stack([v["x"], v["y"], v["z"]], 1).astype(np.float32)
    scales = np.exp(np.stack([v["scale_0"], v["scale_1"], v["scale_2"]], 1).astype(np.float32))
    rots = np.stack([v["rot_0"], v["rot_1"], v["rot_2"], v["rot_3"]], 1).astype(np.float32)
    rots /= (np.linalg.norm(rots, axis=1, keepdims=True) + 1e-9)
    C0 = 0.28209479177387814
    color = np.clip(0.5 + C0 * np.stack([v["f_dc_0"], v["f_dc_1"], v["f_dc_2"]], 1), 0, 1)
    opacity = 1 / (1 + np.exp(-np.asarray(v["opacity"])))
    rgba = np.clip(np.concatenate([color, opacity[:, None]], 1) * 255, 0, 255).astype(np.uint8)
    rot_u8 = np.clip(rots * 128 + 128, 0, 255).astype(np.uint8)
    order = np.argsort(-(opacity * np.prod(scales, 1)))  # important splats first
    with open(splat_path, "wb") as f:
        for i in order:
            f.write(xyz[i].tobytes()); f.write(scales[i].tobytes())
            f.write(rgba[i].tobytes()); f.write(rot_u8[i].tobytes())
    return len(order)

SPLAT_PATH = os.path.join(EXPORT_DIR, f"{scene_name}.splat")
try:
    n = ply_to_splat(PLY_PATH, SPLAT_PATH)
    print(f"  .splat: {SPLAT_PATH}  ({n:,} gaussians, {os.path.getsize(SPLAT_PATH)/1e6:.1f} MB)")
except Exception as e:
    SPLAT_PATH = None; print("  (.splat skipped:", e, ")")

## 9. Download
Zips the results and downloads. View instantly (no install) at
[superspl.at/editor](https://superspl.at/editor) , drag in the `.ply`.

In [ ]:
#@title Zip + download { display-mode: "form" }
import os, zipfile
zip_path = os.path.join(WORK, f"{scene_name}_gaussian_splat.zip")
items = [p for p in (PLY_PATH, SPLAT_PATH) if p and os.path.isfile(p)]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in items: z.write(p, arcname=os.path.basename(p))
    z.writestr("README.txt",
               f"Scene: {scene_name}\nPipeline: Nerfstudio Splatfacto "
               f"({quality_preset}, {train_iterations} iters, sfm={sfm_tool})\n"
               "*.ply = full quality (recommended); *.splat = compact web view.\n"
               "View: https://superspl.at/editor\n")
for p in items: print(f"  {os.path.basename(p):36s} {os.path.getsize(p)/1e6:7.1f} MB")
print(f"zip: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print("Auto-download unavailable:", e, "\nDownload manually from the Files panel:", zip_path)

## Troubleshooting

| Symptom | Fix |
|---|---|
| No GPU (cell 1) | Runtime -> Change runtime type -> GPU, re-run. |
| gsplat compile error (cell 4) | Re-run cell 4 once (kernels cache), or restart runtime + run 1 to 4. |
| Few/no frames in SfM (cell 6) | `sfm_tool='hloc'`; recapture slower, more overlap. |
| CUDA out of memory (cell 7) | `balanced` preset, lower `target_frames`, or L4/A100. |
| Noisy / floaty splat | More `train_iterations`, `high`/`max` preset, more overlapping views. |

**Indoor-geometry upgrade , [DN-Splatter](https://github.com/maturk/dn-splatter):** adds depth+normal
priors on top of nerfstudio and reuses the exact `DATA_DIR` from cell 6. Heavier install / more
version-sensitive, which is why the default is Splatfacto.

**Credits:** Nerfstudio (Tancik et al.), gsplat (Ye et al.), 3D Gaussian Splatting (Kerbl et al.,
SIGGRAPH 2023), COLMAP (Schönberger & Frahm), DN-Splatter (Turkulainen et al., WACV 2025).
